# Portfolio Manager Debug Notebook

This notebook mirrors the portfolio manager pipeline and exposes intermediate outputs for interactive debugging.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## 1) Configure Inputs

Edit these values before running the pipeline.

In [ ]:
# Update these interactively
NUM_FUNDS = 3
RISK_PROFILE = 'moderate'  # conservative | moderate | aggressive
RAW_FUNDS_PATH = Path('mutualfunds/all_funds.tsv')
FUND_INFO_DIR = Path('mutualfunds/fund_info')
FUND_NAMES = [
    'HDFC Mid Cap Dir Gr',
    # 'Add Fund Name 2',
    # 'Add Fund Name 3',
]

print('NUM_FUNDS       :', NUM_FUNDS)
print('RISK_PROFILE    :', RISK_PROFILE)
print('RAW_FUNDS_PATH  :', RAW_FUNDS_PATH)
print('FUND_INFO_DIR   :', FUND_INFO_DIR)
print('FUND_NAMES      :', FUND_NAMES)

## 2) Helpers

These functions are equivalent to your package logic, with extra debug outputs and safer numeric parsing.

In [ ]:
def _to_float(value, default=0.0):
    if pd.isna(value):
        return default
    if isinstance(value, (int, float, np.number)):
        return float(value)
    text = str(value).replace(',', '').strip()
    if text == '':
        return default
    try:
        return float(text)
    except ValueError:
        return default


def load_raw_funds(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t')


def get_fund_isin(fund_name: str, raw_df: pd.DataFrame):
    match = raw_df[raw_df['schemeName'].str.lower() == fund_name.lower()]
    if match.empty:
        return None
    return str(match.iloc[0]['isin'])


def resolve_isins(fund_names, raw_df):
    records = []
    for name in fund_names:
        isin = get_fund_isin(name, raw_df)
        records.append({
            'fund_name': name,
            'isin': isin,
            'resolved': isin is not None
        })
    return pd.DataFrame(records)


def quality_filter_debug(fund_isins, fund_info_dir: Path):
    rows = []
    for isin in fund_isins:
        risk_file = fund_info_dir / f'risk_metrics_{isin}.tsv'
        if not risk_file.exists():
            rows.append({
                'isin': isin,
                'file_found': False,
                'pass_quality': False,
                'reason': 'risk_metrics file missing'
            })
            continue

        risk_df = pd.read_csv(risk_file, sep='\t')
        if risk_df.empty:
            rows.append({
                'isin': isin,
                'file_found': True,
                'pass_quality': False,
                'reason': 'risk_metrics file empty'
            })
            continue

        r = risk_df.iloc[0]
        sharpe_3y = _to_float(r.get('sharpe_3y', 0))
        std_3y = _to_float(r.get('std_3y', 0))
        returns_3y = _to_float(r.get('returns_3y', 0))
        sharpe_cat_avg_3y = _to_float(r.get('sharpe_cat_avg_3y', 0))
        std_cat_avg_3y = _to_float(r.get('std_cat_avg_3y', 0))

        cond_sharpe = sharpe_3y >= sharpe_cat_avg_3y
        cond_std = std_3y <= std_cat_avg_3y * 1.2
        cond_return = returns_3y > 0
        pass_quality = cond_sharpe and cond_std and cond_return

        fail_reasons = []
        if not cond_sharpe:
            fail_reasons.append('sharpe below category avg')
        if not cond_std:
            fail_reasons.append('std too high vs category avg')
        if not cond_return:
            fail_reasons.append('non-positive 3y return')

        rows.append({
            'isin': isin,
            'file_found': True,
            'sharpe_3y': sharpe_3y,
            'sharpe_cat_avg_3y': sharpe_cat_avg_3y,
            'std_3y': std_3y,
            'std_cat_avg_3y': std_cat_avg_3y,
            'returns_3y': returns_3y,
            'pass_quality': pass_quality,
            'reason': '; '.join(fail_reasons) if fail_reasons else 'passed'
        })

    debug_df = pd.DataFrame(rows)
    filtered_isins = debug_df.loc[debug_df['pass_quality'], 'isin'].tolist() if not debug_df.empty else []
    return filtered_isins, debug_df


def rank_funds_debug(filtered_funds, fund_info_dir: Path):
    rows = []
    for isin in filtered_funds:
        risk_file = fund_info_dir / f'risk_metrics_{isin}.tsv'
        if not risk_file.exists():
            continue
        risk_df = pd.read_csv(risk_file, sep='\t')
        if risk_df.empty:
            continue

        r = risk_df.iloc[0]
        sharpe_3y = _to_float(r.get('sharpe_3y', 0))
        sortino_3y = _to_float(r.get('sortino_3y', 0))
        returns_3y = _to_float(r.get('returns_3y', 0))
        std_3y = _to_float(r.get('std_3y', 0))
        score = 0.4 * sharpe_3y + 0.2 * sortino_3y + 0.2 * returns_3y - 0.2 * std_3y

        rows.append({
            'isin': isin,
            'sharpe_3y': sharpe_3y,
            'sortino_3y': sortino_3y,
            'returns_3y': returns_3y,
            'std_3y': std_3y,
            'score': score
        })

    rank_df = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
    ranked_isins = rank_df['isin'].tolist() if not rank_df.empty else []
    return ranked_isins, rank_df


def compute_overlap_matrix_debug(ranked_funds, fund_info_dir: Path):
    holdings = {}
    for isin in ranked_funds:
        holdings_file = fund_info_dir / f'holdings_{isin}.tsv'
        if not holdings_file.exists():
            holdings[isin] = {}
            continue

        df = pd.read_csv(holdings_file, sep='\t')
        if df.empty or 'stock_name' not in df.columns or 'weight' not in df.columns:
            holdings[isin] = {}
            continue

        clean_df = df[['stock_name', 'weight']].copy()
        clean_df['weight'] = clean_df['weight'].map(_to_float)
        clean_df = clean_df.dropna(subset=['stock_name'])

        # Keep latest known weight per stock (more realistic than summing across months).
        stock_weights = clean_df.groupby('stock_name', as_index=False)['weight'].max()
        holdings[isin] = stock_weights.set_index('stock_name')['weight'].to_dict()

    overlap = pd.DataFrame(0.0, index=ranked_funds, columns=ranked_funds)
    pair_rows = []

    for i, a in enumerate(ranked_funds):
        for j in range(i + 1, len(ranked_funds)):
            b = ranked_funds[j]
            all_stocks = set(holdings.get(a, {}).keys()) | set(holdings.get(b, {}).keys())
            overlap_val = 0.0
            for s in all_stocks:
                overlap_val += min(holdings.get(a, {}).get(s, 0.0), holdings.get(b, {}).get(s, 0.0))

            overlap.loc[a, b] = overlap_val
            overlap.loc[b, a] = overlap_val
            pair_rows.append({'fund_a': a, 'fund_b': b, 'overlap_pct': overlap_val})

    pairs_df = pd.DataFrame(pair_rows).sort_values('overlap_pct', ascending=False) if pair_rows else pd.DataFrame(columns=['fund_a', 'fund_b', 'overlap_pct'])
    return overlap, pairs_df


def optimize_portfolio_debug(ranked_funds, overlap_matrix, risk_profile, fund_info_dir: Path):
    risk_profile = str(risk_profile).strip().lower()
    if risk_profile == 'conservative':
        num_select = min(3, len(ranked_funds))
    elif risk_profile == 'moderate':
        num_select = min(4, len(ranked_funds))
    else:
        num_select = min(5, len(ranked_funds))

    selected = ranked_funds[:num_select]

    to_remove = set()
    for i in range(len(selected)):
        for j in range(i + 1, len(selected)):
            if overlap_matrix.loc[selected[i], selected[j]] > 40:
                to_remove.add(selected[j])

    selected = [f for f in selected if f not in to_remove]

    rows = []
    total_ratio = 0.0
    for isin in selected:
        risk_file = fund_info_dir / f'risk_metrics_{isin}.tsv'
        if not risk_file.exists():
            continue
        risk_df = pd.read_csv(risk_file, sep='\t')
        if risk_df.empty:
            continue

        r = risk_df.iloc[0]
        sharpe_3y = _to_float(r.get('sharpe_3y', 0))
        std_3y = _to_float(r.get('std_3y', 0))
        ratio = sharpe_3y / std_3y if std_3y > 0 else 0.0
        total_ratio += ratio
        rows.append({'isin': isin, 'sharpe_3y': sharpe_3y, 'std_3y': std_3y, 'ratio': ratio})

    alloc_df = pd.DataFrame(rows)
    if alloc_df.empty:
        return {}, selected, alloc_df

    if total_ratio > 0:
        alloc_df['weight'] = alloc_df['ratio'] / total_ratio
    else:
        alloc_df['weight'] = 1.0 / len(alloc_df)

    portfolio = dict(zip(alloc_df['isin'], alloc_df['weight']))
    return portfolio, selected, alloc_df.sort_values('weight', ascending=False).reset_index(drop=True)

## 3) Resolve Fund Names to ISINs

In [ ]:
raw_df = load_raw_funds(RAW_FUNDS_PATH)
resolved_df = resolve_isins(FUND_NAMES[:NUM_FUNDS], raw_df)
resolved_df

## 4) Quality Filter (Debug)

In [ ]:
fund_isins = resolved_df.loc[resolved_df['resolved'], 'isin'].tolist()
filtered_isins, quality_debug_df = quality_filter_debug(fund_isins, FUND_INFO_DIR)
print('Input ISIN count   :', len(fund_isins))
print('Passed quality     :', len(filtered_isins))
quality_debug_df

## 5) Ranking (Debug)

In [ ]:
ranked_isins, rank_debug_df = rank_funds_debug(filtered_isins, FUND_INFO_DIR)
print('Ranked funds:', ranked_isins)
rank_debug_df

## 6) Overlap Matrix (Debug)

In [ ]:
overlap_df, overlap_pairs_df = compute_overlap_matrix_debug(ranked_isins, FUND_INFO_DIR)
print('Pairwise overlap details:')
overlap_pairs_df

In [ ]:
print('Overlap matrix (%):')
overlap_df

## 7) Optimize Portfolio (Debug)

In [ ]:
portfolio, selected_isins, allocation_debug_df = optimize_portfolio_debug(
    ranked_isins, overlap_df, RISK_PROFILE, FUND_INFO_DIR
)

print('Selected funds after overlap pruning:', selected_isins)
allocation_debug_df

In [ ]:
final_portfolio_df = pd.DataFrame([
    {'isin': k, 'weight': v} for k, v in portfolio.items()
]).sort_values('weight', ascending=False).reset_index(drop=True) if portfolio else pd.DataFrame(columns=['isin', 'weight'])

final_portfolio_df

## 8) Optional: Map ISIN Back to Fund Name

In [ ]:
if not final_portfolio_df.empty:
    mapping = raw_df[['isin', 'schemeName']].drop_duplicates()
    enriched = final_portfolio_df.merge(mapping, on='isin', how='left')
    enriched = enriched[['schemeName', 'isin', 'weight']]
    enriched
else:
    print('No portfolio generated.')

# Portfolio Manager Interactive Debug Notebook

This notebook converts the `portfolio_manager` logic into a transparent, step-by-step debugging workflow.

## 1) Set Up Notebook Debug Environment

Install/import packages, configure display, logging verbosity, and deterministic seeds for reproducible debugging.

In [ ]:
import logging
import random

import numpy as np
import pandas as pd

np.random.seed(42)
random.seed(42)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger('portfolio_debug')
logger.info('Debug environment ready.')

## 2) Load Portfolio Manager Modules and Enable Reloading

Import the package modules and enable autoreload so edits are immediately available in notebook runs.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import importlib

from portfolio_manager import quality_filter as quality_filter_module
from portfolio_manager import fund_ranker as fund_ranker_module
from portfolio_manager import overlap_matrix as overlap_matrix_module
from portfolio_manager import portfolio_optimizer as portfolio_optimizer_module

logger.info('Modules loaded with autoreload.')

## 3) Build Reproducible Test Fixtures (Portfolio, Prices, Rules)

Define representative fixtures to isolate behavior while debugging pipeline stages.

In [ ]:
FIXTURES = {
    'num_funds': 3,
    'risk_profile': 'moderate',
    'raw_funds_path': Path('mutualfunds/all_funds.tsv'),
    'fund_info_dir': Path('mutualfunds/fund_info'),
    'fund_names': [
        'HDFC Mid Cap Dir Gr',
        # 'Add Fund Name 2',
        # 'Add Fund Name 3',
    ],
    'constraints': {
        'max_pairwise_overlap': 40.0,
        'min_weight': 0.0,
        'max_weight': 1.0,
        'cash_buffer': 0.0,
    },
    'transaction_cost_bps': 15,
}

logger.info('Fixtures loaded: %s', FIXTURES)

## 4) Run the Core Portfolio Update Pipeline Cell-by-Cell

Execute normalization, quality filter, ranking, overlap analysis, and optimization in separate steps.

In [ ]:
# Data normalization and ISIN resolution stage
raw_df = pd.read_csv(FIXTURES['raw_funds_path'], sep='\t')
raw_df.columns = [c.strip() for c in raw_df.columns]

if 'schemeName' not in raw_df.columns or 'isin' not in raw_df.columns:
    raise ValueError('Expected columns schemeName and isin in raw funds file')

norm_df = raw_df[['schemeName', 'isin']].dropna().drop_duplicates().copy()
norm_df['schemeName_norm'] = norm_df['schemeName'].str.strip().str.lower()

selected_names = FIXTURES['fund_names'][: FIXTURES['num_funds']]
resolve_df = pd.DataFrame({'fund_name': selected_names})
resolve_df['fund_name_norm'] = resolve_df['fund_name'].str.strip().str.lower()
resolve_df = resolve_df.merge(
    norm_df[['schemeName_norm', 'isin', 'schemeName']],
    left_on='fund_name_norm',
    right_on='schemeName_norm',
    how='left'
)
resolve_df['resolved'] = resolve_df['isin'].notna()
resolve_df[['fund_name', 'schemeName', 'isin', 'resolved']]

In [ ]:
# Quality filter stage
candidate_isins = resolve_df.loc[resolve_df['resolved'], 'isin'].astype(str).tolist()
filtered_isins = quality_filter_module.quality_filter(candidate_isins)

quality_stage_df = pd.DataFrame({
    'candidate_isin': candidate_isins,
    'passed_quality': [isin in set(filtered_isins) for isin in candidate_isins],
})

print('Input candidates:', len(candidate_isins))
print('Passed quality :', len(filtered_isins))
quality_stage_df

In [ ]:
# Ranking stage
ranked_isins = fund_ranker_module.rank_funds(filtered_isins)
print('Ranked ISINs:', ranked_isins)

rank_df = pd.DataFrame({'isin': ranked_isins})
rank_df['rank'] = range(1, len(rank_df) + 1)
rank_df

In [ ]:
# Overlap stage
overlap_df = overlap_matrix_module.compute_overlap_matrix(ranked_isins)
overlap_df

In [ ]:
# Optimization stage
portfolio = portfolio_optimizer_module.optimize_portfolio(
    ranked_isins,
    overlap_df,
    FIXTURES['risk_profile'],
)

portfolio_df = pd.DataFrame([
    {'isin': k, 'weight': float(v)} for k, v in portfolio.items()
]).sort_values('weight', ascending=False).reset_index(drop=True) if portfolio else pd.DataFrame(columns=['isin', 'weight'])

portfolio_df

## 5) Inspect Intermediate State with Structured Debug Views

Use helper views to compare pre/post stage snapshots and make debugging deterministic.

In [ ]:
def debug_compare(before_df: pd.DataFrame, after_df: pd.DataFrame, key_col: str):
    b = before_df.copy()
    a = after_df.copy()
    b['_stage'] = 'before'
    a['_stage'] = 'after'
    union = pd.concat([b, a], ignore_index=True)
    return union.sort_values([key_col, '_stage']).reset_index(drop=True)


pre_post_quality = debug_compare(
    pd.DataFrame({'isin': candidate_isins}),
    pd.DataFrame({'isin': filtered_isins}),
    key_col='isin'
)
pre_post_quality

In [ ]:
state_snapshot = {
    'input_names': selected_names,
    'resolved_count': int(resolve_df['resolved'].sum()),
    'candidate_count': len(candidate_isins),
    'filtered_count': len(filtered_isins),
    'ranked_count': len(ranked_isins),
    'final_count': len(portfolio_df),
}
state_snapshot

## 6) Trace Rebalancing Decisions and Constraint Checks

Explain why funds are retained/dropped using overlap and exposure diagnostics.

In [ ]:
if overlap_df.empty:
    overlap_diag = pd.DataFrame(columns=['fund_a', 'fund_b', 'overlap_pct', 'decision'])
else:
    rows = []
    for i, a in enumerate(overlap_df.index):
        for j in range(i + 1, len(overlap_df.columns)):
            b = overlap_df.columns[j]
            ov = float(overlap_df.loc[a, b])
            rows.append({
                'fund_a': a,
                'fund_b': b,
                'overlap_pct': ov,
                'decision': 'clip_or_drop' if ov > FIXTURES['constraints']['max_pairwise_overlap'] else 'keep'
            })
    overlap_diag = pd.DataFrame(rows).sort_values('overlap_pct', ascending=False)

overlap_diag

In [ ]:
turnover = float(portfolio_df['weight'].abs().sum()) if not portfolio_df.empty else 0.0
cash_after_alloc = 1.0 - turnover
constraint_diag = {
    'turnover': turnover,
    'cash_after_alloc': cash_after_alloc,
    'min_weight_ok': bool((portfolio_df['weight'] >= FIXTURES['constraints']['min_weight']).all()) if not portfolio_df.empty else True,
    'max_weight_ok': bool((portfolio_df['weight'] <= FIXTURES['constraints']['max_weight']).all()) if not portfolio_df.empty else True,
}
constraint_diag

## 7) Simulate Edge Cases and Failure Paths

Test missing files, invalid risk profile, empty fund list, and unresolved names.

In [ ]:
edge_case_results = {}

# Case 1: Empty fund list
edge_case_results['empty_input'] = quality_filter_module.quality_filter([])

# Case 2: Missing risk file
edge_case_results['missing_risk_file'] = quality_filter_module.quality_filter(['DUMMY_ISIN'])

# Case 3: Invalid risk profile falls back to aggressive branch in current implementation
edge_case_results['invalid_profile_portfolio'] = portfolio_optimizer_module.optimize_portfolio(
    ranked_isins,
    overlap_df,
    'unknown-profile'
)

edge_case_results

## 8) Add Assertion Cells for Regression Checks

Quick checks to detect logic regressions after code changes.

In [ ]:
if not portfolio_df.empty:
    wsum = float(portfolio_df['weight'].sum())
    assert abs(wsum - 1.0) < 1e-6, f'Weights must sum to 1.0, got {wsum}'
    assert (portfolio_df['weight'] >= 0).all(), 'Negative weights found'

if not overlap_df.empty:
    assert (overlap_df.values.diagonal() == 0).all(), 'Diagonal overlap must be zero'

print('Regression assertions passed.')

## 9) Create Interactive Controls for What-If Analysis

Use widgets to vary risk profile and overlap limits, then rerun optimization.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    risk_widget = widgets.Dropdown(
        options=['conservative', 'moderate', 'aggressive'],
        value=FIXTURES['risk_profile'],
        description='Risk:'
    )
    overlap_widget = widgets.FloatSlider(
        value=FIXTURES['constraints']['max_pairwise_overlap'],
        min=0,
        max=100,
        step=1,
        description='Max Overlap'
    )

    display(risk_widget, overlap_widget)

    def run_what_if(risk_profile, max_overlap):
        local_overlap = overlap_df.copy()
        # Keep same logic; diagnostics use this threshold
        local_portfolio = portfolio_optimizer_module.optimize_portfolio(
            ranked_isins,
            local_overlap,
            risk_profile,
        )
        return pd.DataFrame(
            [{'isin': k, 'weight': float(v)} for k, v in local_portfolio.items()]
        ).sort_values('weight', ascending=False).reset_index(drop=True)

    what_if_df = run_what_if(risk_widget.value, overlap_widget.value)
    what_if_df
except Exception as exc:
    print('Widgets unavailable in this kernel/environment:', exc)

## 10) Capture and Export Debug Artifacts

Persist key intermediate tables and final allocations for audit, comparison, and issue reporting.

In [ ]:
output_dir = Path('tuning_results/portfolio_debug')
output_dir.mkdir(parents=True, exist_ok=True)

resolve_df.to_csv(output_dir / 'resolve_df.csv', index=False)
quality_stage_df.to_csv(output_dir / 'quality_stage_df.csv', index=False)
rank_df.to_csv(output_dir / 'rank_df.csv', index=False)
overlap_df.to_csv(output_dir / 'overlap_matrix.csv')
overlap_diag.to_csv(output_dir / 'overlap_diag.csv', index=False)
portfolio_df.to_csv(output_dir / 'portfolio_df.csv', index=False)

print('Artifacts written to', output_dir)

In [ ]:
mapping_df = raw_df[['schemeName', 'isin']].drop_duplicates()
final_named = portfolio_df.merge(mapping_df, on='isin', how='left') if not portfolio_df.empty else pd.DataFrame(columns=['schemeName', 'isin', 'weight'])
final_named[['schemeName', 'isin', 'weight']] if not final_named.empty else final_named